## Carregamento dos Dados e Preparação do Ambiente

### Instalação de Pacotes

In [ ]:
! pip install --upgrade -q xgboost scikit-learn imbalanced-learn matplotlib optuna plotly nbformat

### Importação de Bibliotecas

In [ ]:
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight

from imblearn.over_sampling import SMOTENC
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline

import xgboost as xgb
from xgboost import XGBClassifier

import cupy as cp

import optuna

from utils.constants import *

In [ ]:
df = pd.read_csv("../data/3_gold/dataset-processed-gb.csv")

categorical_features = list(CATEGORICAL_COLUMNS)

for col in categorical_features:
    df[col] = df[col].astype('category')

X = df.drop("class", axis=1)
y = df["class"]
y = y.map(TARGET_LABEL_MAP)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=TEST_RATIO, random_state=RANDOM_STATE, stratify=y
)

In [ ]:
FIXED_PARAMS = {
    "n_estimators": 3000,
    "early_stopping_rounds": 10,
    "enable_categorical": True,
    "objective": "multi:softprob",
    "num_class": N_CLASSES,
    "tree_method": "hist",
    "device": "cuda",
    "eval_metric": "mlogloss",
    "random_state": RANDOM_STATE,
    "verbosity": 0
}

## Hyperparameter Tuning com Optuna

### Definição da Função Objetivo

In [ ]:
def train_on_folds(params, gpu=True):
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    scores = []

    for train_idx, valid_idx in skf.split(X_train_full, y_train_full):
        X_train, y_train = X_train_full[train_idx], y_train_full[train_idx]
        X_valid, y_valid = X_train_full[valid_idx], y_train_full[valid_idx]

        sample_weights = compute_sample_weight(class_weight='balanced', y=y_train)

        if gpu:
            X_train, y_train = cp.asarray(X_train), cp.asarray(y_train)
            X_valid, y_valid = cp.asarray(X_valid), cp.asarray(y_valid)

        model = XGBClassifier(**params)
        model.fit(
            X_train, y_train, 
            eval_set=[(X_valid, y_valid)], 
            sample_weight=sample_weights, 
            verbose=False
        )

        try:
            preds = model.predict(X_valid)
            if gpu: 
                preds = cp.asnumpy(preds)
                y_valid = cp.asnumpy(y_valid)
            f1 = f1_score(y_valid, preds, average='macro')
            scores.append(f1)
        except ValueError:
            return 0.0, 1.0
    
    return np.mean(scores), np.std(scores)


def objective(trial):
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 15),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 1e-1, step=0.01),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 1.0, step=0.1),
        "reg_alpha": trial.suggest_int("reg_alpha", 1, 50),
        "reg_lambda": trial.suggest_int("reg_lambda", 1, 50),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "gamma": trial.suggest_float("gamma", 0, 5, step=0.5),
        **FIXED_PARAMS
    }
    avg, _ = train_on_folds(params, gpu=True)
    return avg

In [ ]:
study = optuna.create_study(study_name="xgboost_study_cuda", direction="maximize")
study.optimize(objective, n_trials=50, timeout=14400, show_progress_bar=True, n_jobs=-1)

best_trial = study.best_trial
print('Best F1:', best_trial.value)
print('Best Params:', best_trial.params)

In [ ]:
import pickle
from pathlib import Path

output_dir = Path('../results/optimization/xgboost')
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / 'optuna_study.pkl', 'wb') as fout:
    pickle.dump(study.sampler, fout)

### Visualizações do Optuna

In [ ]:
import optuna.visualization as vis

display(vis.plot_param_importances(study))
display(vis.plot_optimization_history(study))

## Avaliação Final

In [ ]:
# from sklearn.metrics import classification_report
# import cupy as cp

# # 1. Recuperar os melhores parâmetros do estudo
# final_params = study.best_params
# final_params['tree_method'] = 'hist'
# final_params['device'] = 'cuda'
# final_params['objective'] = 'multi:softprob'
# final_params['num_class'] = NUM_CLASSES
# final_params['eval_metric'] = 'mlogloss'
# final_params['random_state'] = RANDOM_STATE

# # Nota: early_stopping_rounds geralmente não se usa no treino final 
# # a menos que você separe uma validação extra. Vamos confiar nos params do Optuna.

# # -----------------------------------------------------------
# # 2. PREPARAÇÃO DOS DADOS (O Passo que faltava)
# # -----------------------------------------------------------
# print("Preparando dados para o treino final...")

# # A. Normalizar os dados de TREINO (X_opt)
# scaler_final = StandardScaler()
# X_opt_scaled = scaler_final.fit_transform(X_opt) # X_opt são os 560k dados

# # B. Aplicar SMOTE nos dados de TREINO
# smote_final = SMOTE(random_state=42)
# X_opt_resampled, y_opt_resampled = smote_final.fit_resample(X_opt_scaled, y_opt)

# # C. Mandar para GPU
# X_train_final_gpu = cp.asarray(X_opt_resampled)
# y_train_final_gpu = cp.asarray(y_opt_resampled)

# # -----------------------------------------------------------
# # 3. TREINAR O MODELO (O .fit que faltava)
# # -----------------------------------------------------------
# print("Treinando o modelo final...")
# model_final = xgb.XGBClassifier(**final_params)
# model_final.fit(X_train_final_gpu, y_train_final_gpu)

# # -----------------------------------------------------------
# # 4. AVALIAR NO TESTE (O Cofre)
# # -----------------------------------------------------------
# print("Avaliando no conjunto de teste final...")

# # D. Preparar o Teste (X_final_test)
# # IMPORTANTE: 
# # 1. Usar o MESMO scaler treinado no passo A.
# # 2. NÃO usar SMOTE aqui.
# # 3. Mandar para GPU.
# X_test_scaled = scaler_final.transform(X_final_test)
# X_test_gpu = cp.asarray(X_test_scaled)

# # E. Prever
# y_pred_final_test_gpu = model_final.predict(X_test_gpu)
# y_pred_final_test = cp.asnumpy(y_pred_final_test_gpu) # Volta para CPU para o relatório

# # F. Relatório
# print("Relatório de Classificação no Conjunto de Teste:")
# print(classification_report(y_final_test, y_pred_final_test, target_names=target_names, zero_division=0))

In [ ]:
# ConfusionMatrixDisplay.from_predictions(y_final_test, y_pred_final_test, display_labels=target_names, cmap=plt.cm.Blues, normalize='true')

# plt.savefig("optimized_clf_confusion_matrix.pdf")
# plt.show()

In [ ]:
# def plot_metrics_by_class(y_true, y_pred, target_names, figsize=(10, 6)):
#     # Obter report como dicionário
#     report_dict = classification_report(y_true, y_pred, target_names=target_names, 
#                                        zero_division=0, output_dict=True)
    
#     # Criar figura
#     fig, ax = plt.subplots(figsize=figsize)
    
#     x = np.arange(len(target_names))
#     width = 0.25
    
#     # Preparar dados
#     precision_values = [report_dict[cls]['precision'] for cls in target_names]
#     recall_values = [report_dict[cls]['recall'] for cls in target_names]
#     f1_values = [report_dict[cls]['f1-score'] for cls in target_names]
    
#     # Criar barras
#     bars1 = ax.bar(x - width, precision_values, width, label='Precision', alpha=0.8)
#     bars2 = ax.bar(x, recall_values, width, label='Recall', alpha=0.8)
#     bars3 = ax.bar(x + width, f1_values, width, label='F1-Score', alpha=0.8)
    
#     # Adicionar valores no topo das barras
#     for bars in [bars1, bars2, bars3]:
#         for bar in bars:
#             height = bar.get_height()
#             ax.text(bar.get_x() + bar.get_width()/2., height,
#                     f'{height:.2f}',
#                     ha='center', va='bottom', fontsize=9)
    
#     # Configurar gráfico
#     ax.set_xlabel('Classes', fontsize=12)
#     ax.set_ylabel('Values', fontsize=12)
#     ax.set_title('Classification Metrics by Class', fontsize=14)
#     ax.set_xticks(x)
#     ax.set_xticklabels(target_names)
#     ax.legend()
#     ax.set_ylim(0, 1.1)
#     ax.grid(axis='y', alpha=0.3)
    
#     plt.tight_layout()
    
#     return fig, ax, report_dict

In [ ]:
# fig, ax, report_dict = plot_metrics_by_class(y_final_test, y_pred_final_test, target_names=target_names)

# plt.savefig("optimized_clf_metrics_by_class.pdf")
# plt.show()

## Interpretação dos Resultados com SHAP

In [ ]:
# # Using SHAP to explain the optimized model
# explainer = shap.Explainer(best_clf, X_train_resampled)
# shap_values = explainer(X_final_test)

# shap.summary_plot(shap_values, X_final_test, feature_names=feature_names, show=False)
# plt.savefig("optimized_clf_shap_summary_plot.pdf")
# plt.show()